# Modul KKA: Local Search dan Optimasi

## Tujuan praktikum

Praktikum ini membahas hill climbing, simulated annealing, dan genetic algorithm.
Anda akan menentukan representasi solusi, fungsi penilaian, dan cara membangkitkan
tetangga, lalu membandingkan kualitas hasil serta jumlah evaluasinya.

## Persiapan praktikum

Lengkapi identitas berikut. Petunjuk menjalankan Jupyter dan Google Colab tersedia di README.

| Nama | NRP |
|---|---|
| | |

### Mengenali bagian yang harus diisi

Setiap bagian tugas diberi komentar **`# TODO [kode]`**, diikuti penanda
**`________("kode")`** pada posisi jawaban. Nomor pada komentar dan penanda
sama, misalnya U01.

- Ganti **seluruh pemanggilan** `________("U01")`, termasuk tanda kurung dan
  kodenya, dengan ekspresi atau perintah Python yang diminta pada komentar.
- Untuk tugas satu fungsi penuh, tulis implementasi di antara komentar
  `# __ MULAI JAWABAN __` dan `# __ AKHIR JAWABAN __`. Hapus baris penandanya.
- Kode di luar penanda sudah disediakan. Sel bertanda **PENGUJIAN** tidak perlu
  diubah; jalankan setelah fungsi latihan selesai diisi.

Contoh: jika diminta menghitung dua kali x, ubah `nilai = ________("X01")`
menjadi `nilai = 2 * x`.

Jalankan sel berurutan. Setelah mengubah fungsi, jalankan kembali sel definisi
dan sel pengujiannya. Pesan **BELUM DIISI** menyebut kode TODO yang masih kosong.
Jika `assert` gagal, periksa kembali jawaban Anda. Fungsi penanda di bawah
digunakan untuk menampilkan pesan tersebut dan tidak perlu diubah.

In [ ]:
from pathlib import Path
import sys
import random
import numpy as np

# Folder data/ harus berada satu tingkat dengan notebook.
module_root = next((p for p in (Path.cwd(), Path.cwd().parent)
                    if (p / 'data/experiments.py').is_file()), None)
if module_root is None:
    raise FileNotFoundError('Folder data/ tidak ditemukan. Ikuti petunjuk pada README.')
sys.path.insert(0, str(module_root.resolve()))
random.seed(42)
np.random.seed(42)

In [ ]:
# FUNGSI PENANDA LATIHAN — tidak perlu diubah.
def ________(kode):
    raise NotImplementedError(f"TODO [{kode}] belum diisi. Lihat komentar TODO dengan kode yang sama.")

In [ ]:
import matplotlib.pyplot as plt
import math

## 1. Local search dan optimasi

Local search menyimpan solusi saat ini, lalu mencari perbaikan melalui solusi
tetangga. Pada hill climbing dan simulated annealing, pencarian berangkat dari
satu solusi. Genetic algorithm menggunakan sekumpulan solusi atau populasi.

Yang dinilai adalah konfigurasi akhirnya, bukan lintasan menuju konfigurasi itu.
Contohnya, pada penjadwalan, kita membutuhkan jadwal yang memenuhi aturan dan
memiliki nilai baik, tanpa perlu menyimpan urutan perubahan jadwal.

| Istilah | Arti |
|---|---|
| Fungsi objektif / fitness | Mengukur kualitas solusi |
| Tetangga (*neighbor*) | Solusi yang dapat dibentuk melalui satu perubahan yang diizinkan |
| Optimum lokal | Solusi terbaik di antara tetangganya |
| Optimum global | Solusi terbaik pada seluruh ruang pencarian |

Contoh fungsi pada modul ini menggunakan maksimisasi. Pada TSP, jarak tur
diminimalkan; fitness GA ditulis sebagai kebalikan jarak.

## 2. Hill Climbing

### 2.1 Prinsip hill climbing

Hill climbing berpindah ke tetangga yang nilainya lebih baik. Pencarian berhenti
jika tidak ada perbaikan atau batas iterasi tercapai. Metode ini tidak menerima
penurunan nilai, sehingga dapat berhenti pada optimum lokal.

### 2.2 Variasi hill climbing

- **First-choice:** mencoba tetangga sampai menemukan yang lebih baik.
- **Steepest-ascent:** menilai seluruh tetangga dan memilih yang terbaik.
- **Stochastic:** memilih secara acak di antara langkah yang memberikan perbaikan.

Implementasi berikut menggunakan steepest-ascent.

### 2.3 Bentuk ruang pencarian

Untuk fungsi satu variabel, sumbu horizontal menunjukkan nilai x dan sumbu
vertikal menunjukkan $f(x)$. Beberapa bentuk dapat menyulitkan hill climbing:

- **Maksimum lokal:** semua tetangga lebih buruk, padahal ada nilai yang lebih tinggi di tempat lain.
- **Plateau:** beberapa tetangga memiliki nilai yang sama, sehingga tidak ada arah perbaikan langsung.
- **Ridge:** arah perbaikan sulit dicapai dengan bentuk langkah yang tersedia.

Hasil pencarian bergantung pada keadaan awal dan definisi tetangga.

### 2.4 Langkah algoritma

1. Tentukan solusi awal dan hitung nilainya.
2. Bangkitkan serta nilai semua tetangganya.
3. Pindah ke tetangga terbaik jika nilainya lebih besar.
4. Ulangi sampai tidak ada perbaikan atau batas iterasi tercapai.

### 2.5 Batasan

Hill climbing tidak memerlukan seluruh pohon pencarian, tetapi tetap perlu
menilai tetangga. Banyaknya tetangga menentukan biaya satu langkah. Tanpa
langkah menyamping atau restart, algoritma dapat berhenti pada plateau atau
maksimum lokal.

### 2.6 Contoh: maksimisasi fungsi

Gunakan fungsi $f(x)=-x^2+5\cos(2\pi x)$ dengan tetangga $x-0{,}1$ dan $x+0{,}1$.
Maksimum globalnya berada di $x=0$, dengan $f(0)=5$, karena $-x^2\le0$ dan
$\cos(2\pi x)\le1$.

Definisikan fungsi objektif dan pembangkit tetangga berikut.

In [ ]:
def objective(x):
    return -(x**2) + 5 * np.cos(2 * np.pi * x)

def generate_neighbors(x, step_size=0.1):
    return [np.array([x[0] + step_size]), np.array([x[0] - step_size])]

Implementasikan perpindahan ke tetangga terbaik.

In [ ]:
def hill_climbing(objective, initial, n_iterations=100, step_size=0.1):
    current = np.array([initial])
    
    current_eval = objective(current).item() 
    history = [(current[0], current_eval)]

    for i in range(n_iterations):
        neighbors = generate_neighbors(current, step_size)
        
        neighbor_evals = [objective(n).item() for n in neighbors] 
        best_idx = np.argmax(neighbor_evals) 

        if neighbor_evals[best_idx] > current_eval: 
            current = neighbors[best_idx]
            current_eval = neighbor_evals[best_idx]
            
            history.append((current[0], current_eval))
            print(f"Step {i+1}: x = {current[0]:.4f}, f(x) = {current_eval:.4f}")
        else:
            print("Tidak ada tetangga yang lebih baik. Algoritma konvergen.")
            break

    return current, current_eval, history

Jalankan dari `initial=2.0`. Pada nilai ini, kedua tetangga berjarak 0,1
memiliki nilai lebih rendah, sehingga algoritma langsung berhenti dengan $f=1$.
Bandingkan dengan `initial=0.3`, yang dapat bergerak mendekati maksimum global.

In [ ]:
initial_guess = 2.0
solution, value, history = hill_climbing(objective, initial_guess, n_iterations=100, step_size=0.1)
print(f"\nSolusi terbaik: x = {solution[0]:.4f}, f(x) = {value:.4f}")
solution_global, value_global, _ = hill_climbing(objective, 0.3)
assert abs(value_global - 5) < 1e-8
print('Start 0.3 mendekati global:', solution_global, value_global)

Gambarkan fungsi dan lintasan nilai x selama pencarian.

In [ ]:
x_range = np.linspace(-4, 4, 200)

y_range = [-(x**2) + 5 * np.cos(2 * np.pi * x) for x in x_range]

hx = [h[0] for h in history]
hy = [h[1] for h in history]

plt.figure(figsize=(10, 5))

plt.plot(x_range, y_range, 'b-', label='f(x) = -x^2 + 5 cos(2pi x)')
plt.plot(hx, hy, 'ro-', label='Jejak Hill Climbing', markersize=6)
plt.plot(hx[0], hy[0], 'gs', markersize=12, label='Start')
plt.plot(hx[-1], hy[-1], 'r*', markersize=15, label='Solusi')
plt.xlabel('x')
plt.ylabel('f(x)')

plt.title('Hill Climbing pada f(x) = -x^2 + 5 cos(2pi x)')
plt.legend()
plt.grid(True)
plt.show()

## 3. Simulated Annealing

### 3.1 Prinsip simulated annealing

Simulated annealing (SA) kadang menerima solusi yang nilainya lebih buruk agar
pencarian dapat keluar dari optimum lokal. Peluang penerimaan diatur oleh suhu T.
Saat suhu turun, peluang menerima penurunan nilai juga mengecil.

### 3.2 Aturan penerimaan untuk maksimisasi

1. Mulai dari solusi S dan suhu positif T.
2. Bangkitkan tetangga $S'$ dan hitung $\Delta=f(S')-f(S)$.
3. Terima jika $\Delta\ge0$. Jika $\Delta<0$, terima dengan peluang $e^{\Delta/T}$.
4. Kurangi suhu dan ulangi sampai batas iterasi tercapai.
5. Simpan solusi terbaik yang pernah ditemukan, karena solusi terakhir belum tentu yang terbaik.

Untuk minimisasi, tanda aturan penerimaan dibalik. Contoh di sini menggunakan
maksimisasi secara konsisten.

### 3.3 Jadwal pendinginan

Contoh ini menggunakan $T_{baru}=\alpha T$ dengan $T_0=10$ dan $\alpha=0{,}995$.
Nilai alpha yang mendekati 1 membuat suhu turun lebih lambat.

Jika suhu turun terlalu cepat, kesempatan keluar dari optimum lokal berkurang.
Jika terlalu lambat, banyak iterasi masih menerima penurunan nilai. Pilih suhu,
alpha, dan batas iterasi berdasarkan hasil percobaan.

### 3.4 Batasan

SA dapat menerima langkah yang tidak akan diambil hill climbing, tetapi hasil
dengan jumlah iterasi terbatas tidak dijamin optimal. Hasilnya dipengaruhi
suhu awal, jadwal pendinginan, tetangga, dan bilangan acak yang digunakan.

### 3.5 Perbandingan dengan hill climbing

| Aspek | Hill climbing pada contoh | Simulated annealing |
|---|---|---|
| Pemilihan tetangga | Terbaik dari seluruh tetangga | Satu tetangga acak |
| Langkah yang menurunkan nilai | Ditolak | Dapat diterima dengan peluang $e^{\Delta/T}$ |
| Keadaan saat berhenti | Dapat berupa optimum lokal | Dapat berbeda dari solusi terbaik yang pernah ditemukan |
| Parameter utama | Keadaan awal dan ukuran langkah | Keadaan awal, ukuran langkah, suhu, alpha, dan batas iterasi |

### 3.6 Contoh: fungsi objektif yang sama

Gunakan kembali bentuk fungsi pada contoh hill climbing.

In [ ]:
def objective_sa(x):
    return -(x**2) + 5 * np.cos(2 * np.pi * x)

Implementasikan aturan penerimaan dan simpan solusi terbaik selama pencarian.

In [ ]:
def simulated_annealing(objective, initial, T=10, alpha=0.995, n_iterations=2000, step_size=0.5):
    current = initial
    current_eval = objective(current)
    best = current
    best_eval = current_eval
    history = [(current, current_eval, T)]

    for i in range(n_iterations):
        # Generate tetangga random
        neighbor = current + random.uniform(-step_size, step_size)
        neighbor_eval = objective(neighbor)

        # Hitung delta
        delta = neighbor_eval - current_eval

        # Acceptance rule untuk MAXIMIZATION
        if delta > 0:
            current = neighbor
            current_eval = neighbor_eval
            if current_eval > best_eval:
                best = current
                best_eval = current_eval
        else:
            # Probabilitas menerima solusi yang lebih buruk (delta bernilai negatif)
            prob = math.exp(delta / T)
            if random.random() < prob:
                current = neighbor
                current_eval = neighbor_eval

        # Cooling schedule
        T = T * alpha
        history.append((current, current_eval, T))

        if (i + 1) % 200 == 0:
            print(f"Step {i+1}: x = {current:.4f}, f(x) = {current_eval:.4f}")

        if T < 1e-10:
            break

    return best, best_eval, history

Jalankan dari x=2, lalu tampilkan solusi terbaiknya.

In [ ]:
best, best_eval, history_sa = simulated_annealing(objective_sa, initial=2.0)
print(f"Solusi terbaik: x = {best:.4f}, f(x) = {best_eval:.4f}")

Gambarkan perubahan nilai fungsi dan suhu pada setiap iterasi.

In [ ]:
iters = range(len(history_sa))
vals = [h[1] for h in history_sa]

# Ekstraksi suhu dari history, dengan rumus fallback jika data tidak tersedia

temps = [h[2] for i, h in enumerate(history_sa)]

fig, ax1 = plt.subplots(figsize=(10, 5))

# Konfigurasi Sumbu Y Kiri untuk f(x)
color_fx = 'tab:blue'
ax1.set_xlabel('Iterasi')
ax1.set_ylabel('f(x)', color=color_fx)
ax1.plot(iters, vals, color=color_fx, alpha=0.6, label='Nilai f(x) per iterasi')
ax1.axhline(y=best_eval, color='r', linestyle='--', label=f'Terbaik: f(x)={best_eval:.4f}')
ax1.tick_params(axis='y', labelcolor=color_fx)
ax1.grid(True)

# Konfigurasi Sumbu Y Kanan untuk Suhu
ax2 = ax1.twinx()  
color_temp = 'tab:red'
ax2.set_ylabel('Suhu (T)', color=color_temp)  
ax2.plot(iters, temps, color=color_temp, alpha=0.8, label='Penurunan Suhu')
ax2.tick_params(axis='y', labelcolor=color_temp)

fig.tight_layout()  
plt.title('Konvergensi Simulated Annealing dan Perubahan Suhu')

# Menggabungkan legenda dari kedua sumbu agar rapi
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='lower right')

plt.show()

## 4. Genetic Algorithm

### 4.1 Prinsip genetic algorithm

Genetic algorithm (GA) memperbaiki populasi solusi melalui seleksi, crossover,
dan mutasi. Individu dengan fitness lebih baik lebih berpeluang dipilih sebagai
induk. Keturunannya membentuk generasi berikutnya.

### 4.2 Representasi dan elitism

| Istilah | Arti |
|---|---|
| Populasi | Kumpulan solusi pada satu generasi |
| Kromosom | Representasi satu solusi |
| Gen | Satu bagian atau variabel dalam kromosom |
| Fitness | Nilai kualitas kromosom |

Pada optimasi satu variabel, kromosom dapat berupa satu bilangan real. Pada TSP,
kromosom berupa urutan kota tanpa pengulangan.

**Elitism** mempertahankan beberapa individu terbaik tanpa mutasi. Contoh ini
menyimpan dua individu terbaik dan menempatkannya pada generasi berikutnya.
Cara ini mencegah fitness terbaik menurun, tetapi terlalu banyak elite dapat
mengurangi keragaman populasi.

### 4.3 Operator yang digunakan

- **Tournament selection:** ambil beberapa individu secara acak, lalu pilih yang fitness-nya tertinggi.
- **Arithmetic crossover:** gabungkan dua nilai real melalui rata-rata berbobot.
- **Order crossover (OX1):** salin segmen urutan dari satu induk, lalu isi sisa kota menurut urutan induk lain.
- **Mutasi:** beri perubahan acak pada nilai real, atau tukar dua kota untuk kromosom TSP.

Operator harus sesuai dengan representasi solusi. Crossover pada TSP, misalnya,
harus tetap menghasilkan setiap kota tepat satu kali.

### 4.4 Langkah algoritma

1. Bangkitkan populasi awal dan hitung fitness-nya.
2. Simpan individu elite.
3. Pilih induk, lakukan crossover, lalu mutasi.
4. Bentuk generasi baru dengan menyertakan elite.
5. Ulangi sampai batas generasi atau kriteria berhenti tercapai.
6. Kembalikan individu terbaik.

### 4.5 Batasan

GA tidak memerlukan turunan fungsi, tetapi menilai banyak individu pada setiap
generasi. Hasilnya bergantung pada ukuran populasi, operator, peluang mutasi,
dan jumlah generasi. Populasi dapat kehilangan keragaman sebelum optimum global ditemukan.

### 4.6 Contoh: optimasi satu variabel

Maksimalkan $f(x)=x\sin(10\pi x)+1$ pada interval $[-1,2]$.
Fungsi ini memiliki beberapa puncak lokal. Amati perubahan fitness terbaik
dan rata-rata populasi.

In [ ]:
def fitness_function(x):
    return x * np.sin(10 * np.pi * x) + 1

Tetapkan parameter dan bangkitkan populasi awal.

In [ ]:
POP_SIZE = 40
GENERATIONS = 1000
X_MIN, X_MAX = -1.0, 2.0
CROSSOVER_PROB = 0.9
MUTATION_PROB = 0.2
MUTATION_STD = 0.1

np.random.seed(42)
population = np.random.uniform(X_MIN, X_MAX, POP_SIZE)

Definisikan seleksi, crossover, dan mutasi untuk kromosom bilangan real.

In [ ]:
def tournament_selection(pop, fitness, k=3):
    selected = []
    for _ in range(len(pop)):
        idx = np.random.choice(len(pop), k, replace=False)
        selected.append(pop[idx[np.argmax(fitness[idx])]])
    return np.array(selected)

def arithmetic_crossover(p1, p2):
    alpha = np.random.rand()
    return alpha * p1 + (1 - alpha) * p2, alpha * p2 + (1 - alpha) * p1

def mutate(x):
    if np.random.rand() < MUTATION_PROB:
        x += np.random.normal(0, MUTATION_STD)
    return np.clip(x, X_MIN, X_MAX)

Jalankan evolusi dengan dua individu elite pada setiap generasi.

In [ ]:
best_history = []
mean_history = []

for gen in range(GENERATIONS):
    fitness = fitness_function(population)
    best_history.append(np.max(fitness))
    mean_history.append(np.mean(fitness))

    best_idx_gen = np.argmax(fitness)
    if (gen + 1) % 100 == 0:
        print(f"Step {gen+1}: x terbaik = {population[best_idx_gen]:.4f}, f(x) = {fitness[best_idx_gen]:.4f}")

    parents = tournament_selection(population, fitness)
    elites = population[np.argsort(fitness)[-2:]].copy()
    offspring = []
    np.random.shuffle(parents)

    for i in range(0, POP_SIZE, 2):
        if np.random.rand() < CROSSOVER_PROB:
            c1, c2 = arithmetic_crossover(parents[i], parents[i + 1])
        else:
            c1, c2 = parents[i], parents[i + 1]
        offspring.extend([mutate(c1), mutate(c2)])

    population = np.array(offspring)
    offspring_fitness = fitness_function(population)
    population[np.argsort(offspring_fitness)[:2]] = elites

# Hasil akhir
final_fitness = fitness_function(population)
best_idx = np.argmax(final_fitness)
print(f"Solusi terbaik: x = {population[best_idx]:.4f}, f(x) = {final_fitness[best_idx]:.4f}")
assert all(b >= a - 1e-12 for a, b in zip(best_history, best_history[1:]))
print('LULUS elitism: fitness terbaik tidak menurun')

Tampilkan perubahan fitness dan sebaran populasi akhir.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Konvergensi fitness
axes[0].plot(best_history, label='Fitness terbaik')
axes[0].plot(mean_history, label='Rata-rata fitness')
axes[0].set_xlabel('Generasi')
axes[0].set_ylabel('Fitness')
axes[0].set_title('Konvergensi GA')
axes[0].legend()
axes[0].grid(True)

# Plot 2: Populasi akhir pada fungsi objektif
x = np.linspace(X_MIN, X_MAX, 500)
axes[1].plot(x, fitness_function(x), 'b-', label='f(x)')
axes[1].scatter(population, fitness_function(population), c='red', zorder=5, label='Populasi Akhir')
axes[1].scatter(population[best_idx], final_fitness[best_idx], c='gold', s=200, marker='*', zorder=6, label='Terbaik')
axes[1].set_xlabel('x')
axes[1].set_ylabel('f(x)')
axes[1].set_title('Populasi Akhir pada Objective Function')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 5. Perbandingan metode

| Aspek | Hill climbing | Simulated annealing | Genetic algorithm |
|---|---|---|---|
| Solusi yang disimpan | Satu solusi saat ini | Solusi saat ini dan solusi terbaik | Populasi |
| Pemilihan kandidat pada contoh | Seluruh tetangga | Satu tetangga acak | Seleksi dan pembentukan keturunan |
| Cara menghadapi optimum lokal | Dapat menggunakan restart | Menerima penurunan nilai secara probabilistik | Menggunakan keragaman, crossover, dan mutasi |
| Biaya evaluasi | Per tetangga yang diperiksa | Per proposal tetangga | Per individu yang dinilai |
| Jaminan optimum global dengan iterasi terbatas | Tidak | Tidak | Tidak |

Riwayat pencarian dan daftar tetangga pada kode visualisasi membutuhkan memori
tambahan. Saat membandingkan metode, catat kualitas solusi, jumlah evaluasi,
serta waktu; jangan hanya melihat jumlah iterasi.

## 6. Latihan dasar

Lengkapi bagian penanda `________("kode")` pada tiga latihan berikut. Jalankan pengujian
setelah setiap fungsi selesai. Gunakan hasil percobaan untuk menjawab pertanyaan.

### Latihan 1: hill climbing untuk 8-queens

Tempatkan delapan ratu pada papan 8 × 8 tanpa saling menyerang. Representasikan
keadaan sebagai satu posisi baris untuk setiap kolom. Satu tetangga memindahkan
satu ratu ke baris lain dalam kolom yang sama. Fitness adalah jumlah pasangan
ratu yang tidak saling menyerang; nilai maksimumnya 28.

Pertanyaan:
1. Mengapa hill climbing dapat berhenti sebelum fitness mencapai 28?
2. Apa pengaruh mengizinkan langkah dengan fitness yang sama (*sideways move*)?

In [ ]:
# BAGIAN TUGAS — isi 1 penanda: L01.
try:
    def count_non_attacking(queens):
        """Hitung jumlah pasangan ratu yang tidak saling menyerang."""
        n = len(queens)
        non_attacking = 0
        for i in range(n):
            for j in range(i+1, n):
                # TODO [L01]: Cek bahwa kedua ratu tidak satu diagonal.
                if queens[i] != queens[j] and ________("L01"):
                    # Cek bukan di diagonal yang sama
                    non_attacking += 1
        return non_attacking

    def get_neighbors_queens(queens):
        """Generate semua neighbor: pindahkan 1 ratu ke baris lain di kolomnya."""
        neighbors = []
        n = len(queens)
        for col in range(n):
            for row in range(n):
                if queens[col] != row:
                    new_queens = list(queens)
                    new_queens[col] = row
                    neighbors.append(tuple(new_queens))
        return neighbors

    def hill_climbing_queens(n=8, max_iter=1000, initial=None):
        current = tuple(initial) if initial is not None else tuple(random.randint(0, n-1) for _ in range(n))
        current_fitness = count_non_attacking(current)
        max_fitness = n * (n - 1) // 2  # semua pasangan tidak menyerang

        for i in range(max_iter):
            if current_fitness == max_fitness:
                return current, current_fitness, True

            neighbors = get_neighbors_queens(current)
            best = max(neighbors, key=count_non_attacking)
            best_fitness = count_non_attacking(best)

            if best_fitness <= current_fitness:
                return current, current_fitness, False  # terjebak

            current = best
            current_fitness = best_fitness

        return current, current_fitness, False
except NotImplementedError as error:
    print("BELUM DIISI:", error)

In [ ]:
# PENGUJIAN — jalankan setelah fungsi latihan diisi; tidak perlu diubah.
try:
    sol, fit, success = hill_climbing_queens(8)
    print(f"Solusi: {sol}, Fitness: {fit}, Berhasil: {success}")
    assert count_non_attacking((0, 4, 7, 5, 2, 6, 1, 3)) == 28
    assert len(get_neighbors_queens((0,)*8)) == 56
    assert success == (fit == 28)
    print('LULUS HC N-Queens')
except NotImplementedError as error:
    print("BELUM DIISI:", error)

### Latihan 2: simulated annealing untuk 8-queens

Lengkapi `sa_queens` dan bandingkan hasilnya dengan hill climbing pada 20 pasang
keadaan awal yang sama. Gunakan aturan penerimaan untuk maksimisasi. Hitung
berapa percobaan yang mencapai fitness 28.

Pertanyaan:
1. Mengapa kedua metode perlu memulai dari keadaan yang sama?
2. Apakah lebih banyak percobaan berhasil berarti waktu pencarian lebih singkat?

In [ ]:
# BAGIAN TUGAS — isi 1 penanda: L02.
try:
    def sa_queens(n=8, T=4.0, alpha=0.99, n_iter=5000, initial=None):
        current = list(initial) if initial is not None else [random.randrange(n) for _ in range(n)]
        current_fitness = count_non_attacking(current)
        max_fitness = n * (n-1) // 2

        for i in range(n_iter):
            if current_fitness == max_fitness:
                return tuple(current), current_fitness, True

            new = current[:]
            col = random.randint(0, n-1)
            new[col] = random.randint(0, n-1)  # random row baru
            new_fitness = count_non_attacking(new)

            delta = new_fitness - current_fitness
            # TODO [L02]: Isi eksponen peluang penerimaan berdasarkan delta dan T.
            if delta > 0 or (T > 0 and random.random() < math.exp(________("L02"))):
                current = new
                current_fitness = new_fitness

            T *= alpha

        return tuple(current), current_fitness, current_fitness == max_fitness
except NotImplementedError as error:
    print("BELUM DIISI:", error)

In [ ]:
# PENGUJIAN — jalankan setelah fungsi latihan diisi; tidak perlu diubah.
try:
    # Perbandingan fungsi latihan pada 20 pasang start yang sama.
    sa_success = hc_success = 0
    for i in range(20):
        seed = 42 + i
        rng = random.Random(seed)
        initial = tuple(rng.randrange(8) for _ in range(8))
        random.seed(seed)
        sa_success += sa_queens(initial=initial)[2]
        hc_success += hill_climbing_queens(initial=initial)[2]
    print(f'Hasil 20 percobaan SA: {sa_success}/20, HC: {hc_success}/20')
    assert 0 <= sa_success <= 20 and 0 <= hc_success <= 20
    print('LULUS pengujian 20 percobaan')
except NotImplementedError as error:
    print("BELUM DIISI:", error)

### Latihan 3: genetic algorithm untuk TSP

Cari tur tertutup melalui enam kota menggunakan GA. Setiap kota harus dikunjungi
tepat satu kali sebelum kembali ke kota awal. Gunakan kromosom permutasi,
crossover OX1, dan mutasi pertukaran dua kota.

Pertanyaan:
1. Mengapa crossover yang memotong dan menggabungkan list biasa dapat menggandakan kota?
2. Bagaimana elitism mempertahankan tur terbaik yang sudah ditemukan?

In [ ]:
# BAGIAN TUGAS — isi 2 penanda: L03, L04.
try:
    cities = {'A': (0, 0), 'B': (2, 0), 'C': (3, 2), 'D': (2, 4), 'E': (0, 3), 'F': (-1, 1)}

    def total_distance(route, cities):
        # Tur tertutup: kota terakhir kembali ke kota pertama.
        return sum(math.dist(cities[a], cities[b]) for a, b in zip(route, route[1:] + route[:1]))

    def order_crossover(p1, p2):
        n = len(p1)
        i, j = sorted(random.sample(range(n), 2))
        child = [None] * n
        # TODO [L03]: Salin segmen parent pertama.
        child[i:j+1] = ________("L03")  # segment dari p1
        fill = [p2[(j + 1 + k) % n] for k in range(n) if p2[(j + 1 + k) % n] not in child]
        positions = [(j + 1 + k) % n for k in range(n) if child[(j + 1 + k) % n] is None]
        for k, gene in zip(positions, fill):
            child[k] = gene
        return child

    def swap_mutation(route, prob=0.1):
        if random.random() < prob:
            i, j = random.sample(range(len(route)), 2)
            # TODO [L04]: Tukar nilai route[i] dan route[j] secara bersamaan.
            route[i], route[j] = ________("L04")
        return route

    def ga_tsp(cities, pop_size=50, gens=200):
        names = list(cities.keys())
        pop = [random.sample(names, len(names)) for _ in range(pop_size)]

        for gen in range(gens):
            fitnesses = [1 / total_distance(r, cities) for r in pop]
            new_pop = [r[:] for r in sorted(pop, key=lambda r: total_distance(r, cities))[:2]]
            for _ in range((pop_size - 2) // 2):
                p1 = max(random.sample(pop, 3), key=lambda r: 1/total_distance(r, cities))
                p2 = max(random.sample(pop, 3), key=lambda r: 1/total_distance(r, cities))
                c1 = order_crossover(p1, p2)
                c2 = order_crossover(p2, p1)
                new_pop.extend([swap_mutation(c1), swap_mutation(c2)])
            pop = new_pop

        best = min(pop, key=lambda r: total_distance(r, cities))
        return best, total_distance(best, cities)
except NotImplementedError as error:
    print("BELUM DIISI:", error)

In [ ]:
# PENGUJIAN — jalankan setelah fungsi latihan diisi; tidak perlu diubah.
try:
    random.seed(42)
    route, dist = ga_tsp(cities)
    print(f"GA TSP Route: {route}, Distance: {dist:.2f}")
    from itertools import permutations
    assert set(route) == set(cities) and len(route) == len(cities)
    assert math.isclose(dist, total_distance(route, cities))
    exact_distance = min(total_distance(list(p), cities) for p in permutations(cities))
    print('Optimal brute-force 6 kota:', exact_distance, 'gap GA:', dist-exact_distance)
    assert dist >= exact_distance - 1e-9
    print('LULUS GA TSP: tur valid dan biaya konsisten')
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 7. Random restart dan perbandingan tingkat keberhasilan

Random restart menjalankan hill climbing dari beberapa keadaan awal acak dan
menyimpan hasil terbaik. Jika peluang sukses satu percobaan adalah p dan setiap
percobaan independen, peluang setidaknya satu sukses dalam R percobaan adalah
$1-(1-p)^R$. Jumlah evaluasi juga bertambah.

Bandingkan HC dan SA pada **300 pasang keadaan awal yang sama** untuk 8-queens.
Percobaan ke-i menggunakan seed `42+i`. HC tidak menerima langkah menyamping;
SA menggunakan T=10, alpha=0,995, dan paling banyak 2.000 langkah. Percobaan
berhasil jika fitness mencapai 28.

HC menilai 56 tetangga pada satu langkah, sedangkan SA menilai satu proposal.
Catat tingkat keberhasilan dan jumlah evaluasi, lalu bandingkan dengan hasil
20 percobaan pada latihan sebelumnya.

In [ ]:
from data.experiments import compare_local, restart_hc, queen_fitness
from time import perf_counter
before = perf_counter()
local_rows = compare_local(runs=300, seed=42)
for method in ('HC', 'SA'):
    rows = [r for r in local_rows if r['method'] == method]
    successes = sum(r['success'] for r in rows)
    assert all(r['score'] == queen_fitness(r['state']) for r in rows)
    print(method, f'{successes}/300 ({100*successes/300:.1f}%)',
          'rata-rata evaluasi', round(sum(r['evaluations'] for r in rows)/300, 2))
print('Waktu seluruh percobaan:', round(perf_counter()-before, 3), 'detik')
restart_result = restart_hc(restarts=20, seed=42)
assert restart_result['attempts'] <= 20
print('Random restart:', restart_result)
print('LULUS perbandingan HC dan SA')

### Latihan lanjutan: parameter SA dan elitism GA

Uji alpha SA sebesar 0,99; 0,995; dan 0,999. Untuk GA, bandingkan 0, 1, dan 2
individu elite. Gunakan seed serta anggaran evaluasi yang sama sejauh memungkinkan.
Catat tingkat keberhasilan atau selisih biaya terhadap optimum, serta waktunya.
Jelaskan pengaruh pendinginan dan jumlah elite pada hasil.

### Tugas mandiri: random restart

Tulis `random_restart_student` dengan batas jumlah restart. Bangkitkan keadaan
awal baru pada setiap percobaan, simpan hasil terbaik, dan berhenti lebih awal
jika fitness mencapai 28. Gunakan seed tetap untuk pengujian.

In [ ]:
# BAGIAN TUGAS — isi 1 penanda: L05.
try:
    def random_restart_student(restarts=20, seed=42):
        # TODO [L05]: Tulis seluruh isi random_restart_student: ulangi HC dari keadaan awal baru dan simpan hasil terbaik.
        # __ MULAI JAWABAN __
        ________("L05")
        # __ AKHIR JAWABAN __

    # PENGUJIAN — jalankan setelah fungsi di atas diisi; tidak perlu diubah.
    restart_answer = random_restart_student()
    assert restart_answer['score'] == queen_fitness(restart_answer['state'])
    assert restart_answer['score'] == 28
    print('LULUS random restart')
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## Referensi

- Russell, S. & Norvig, P. *Artificial Intelligence: A Modern Approach*, edisi ke-4, Bab 4.
  [Daftar isi](https://aima.cs.berkeley.edu/contents.html).
- [Kode pendamping AIMA](https://github.com/aimacode/aima-python).